# SQL Agent with Observability using MLFlow

`pyproject.toml`:
```toml
[project]
name = "sql-agent-mflow"
version = "0.1.0"
requires-python = ">=3.13"
dependencies = [
    "ipykernel>=7.2.0",
    "langchain>=1.3.2",
    "langchain-nvidia-ai-endpoints>=1.4.0",
    "langchain-openai>=1.2.2",
    "langgraph>=1.2.2",
    "mlflow>=3.12.0",
]
```

In [1]:
import os
from langchain.chat_models import init_chat_model
import mlflow

/Users/vasim/Programming/ai-engineering/18-sql-agent-mlflow/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
API_KEY = os.environ["NVIDIA_API_KEY"]
BASE_URL = "https://integrate.api.nvidia.com/v1"
CHAT_MODEL_NAME = "openai/gpt-oss-120b"

In [3]:
llm = init_chat_model(CHAT_MODEL_NAME, model_provider="nvidia", base_url=BASE_URL, api_key=API_KEY)

In [4]:
llm.invoke("Ping")

AIMessage(content='Pong!', additional_kwargs={'reasoning_content': 'The user wrote "Ping". Likely they want a response of "Pong" or something acknowledging. Could be a test. I\'ll respond with "Pong".', 'reasoning': 'The user wrote "Ping". Likely they want a response of "Pong" or something acknowledging. Could be a test. I\'ll respond with "Pong".', '_reasoning_api_fields': ['reasoning_content', 'reasoning']}, response_metadata={'role': 'assistant', 'content': 'Pong!', 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [], 'reasoning': 'The user wrote "Ping". Likely they want a response of "Pong" or something acknowledging. Could be a test. I\'ll respond with "Pong".', 'reasoning_content': 'The user wrote "Ping". Likely they want a response of "Pong" or something acknowledging. Could be a test. I\'ll respond with "Pong".', 'token_usage': {'prompt_tokens': 66, 'total_tokens': 112, 'completion_tokens': 46, 'prompt_tokens_details': None}, 'finish_rea

In [5]:
import mlflow

- Becareful while assigning a port to MLFlow server, if same port is used already, MLFlow will throw errors

In [6]:
TRACKING_URI = "http://localhost:5909"

In [7]:
mlflow.set_tracking_uri(TRACKING_URI)

In [8]:
print(mlflow.get_tracking_uri())

http://localhost:5909


In [9]:
# Specify the experiment you just created for your LLM application or AI agent.
mlflow.set_experiment("sql-agent")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1780148195862, experiment_id='1', last_update_time=1780148195862, lifecycle_stage='active', name='sql-agent', tags={}, trace_location=None, workspace='default'>

In [10]:
@mlflow.trace
def ask_ai(query: str, user_id: str, session_id: str):
    """Ask AI any query"""
    mlflow.update_current_trace(
        metadata={
            "mlflow.trace.user": user_id,
            "mlflow.trace.session": session_id
        }
    )
    return llm.invoke("Ping")

In [11]:
print(ask_ai("Hello", "123", "456"))

content='Pong!' additional_kwargs={'reasoning_content': 'The user just sent "Ping". Likely expecting a "Pong" response. Could be a test. I\'ll respond with "Pong".', 'reasoning': 'The user just sent "Ping". Likely expecting a "Pong" response. Could be a test. I\'ll respond with "Pong".', '_reasoning_api_fields': ['reasoning_content', 'reasoning']} response_metadata={'role': 'assistant', 'content': 'Pong!', 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [], 'reasoning': 'The user just sent "Ping". Likely expecting a "Pong" response. Could be a test. I\'ll respond with "Pong".', 'reasoning_content': 'The user just sent "Ping". Likely expecting a "Pong" response. Could be a test. I\'ll respond with "Pong".', 'token_usage': {'prompt_tokens': 66, 'total_tokens': 108, 'completion_tokens': 42, 'prompt_tokens_details': None}, 'finish_reason': 'stop', 'model_name': 'openai/gpt-oss-120b'} id='lc_run--019e7bac-0f5b-7fb3-abec-fe73cbd3f96e-0' tool_calls=[]

In [12]:
@mlflow.trace
def ask_ai(query: str):
    """Ask AI any query"""
    return llm.invoke(query)

In [13]:
ask_ai("Hi")

AIMessage(content='Hello! 👋 How can I assist you today?', additional_kwargs={'reasoning_content': 'We need to respond as ChatGPT. The user just said "Hi". We should greet back and ask how we can help. Possibly ask for clarification. We can be friendly.', 'reasoning': 'We need to respond as ChatGPT. The user just said "Hi". We should greet back and ask how we can help. Possibly ask for clarification. We can be friendly.', '_reasoning_api_fields': ['reasoning_content', 'reasoning']}, response_metadata={'role': 'assistant', 'content': 'Hello! 👋 How can I assist you today?', 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [], 'reasoning': 'We need to respond as ChatGPT. The user just said "Hi". We should greet back and ask how we can help. Possibly ask for clarification. We can be friendly.', 'reasoning_content': 'We need to respond as ChatGPT. The user just said "Hi". We should greet back and ask how we can help. Possibly ask for clarification. We

In [14]:
mlflow.langchain.autolog()

In [15]:
import requests, pathlib

url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"
local_path = pathlib.Path("Chinook.db")

if local_path.exists():
    print(f"{local_path} already exists, skipping download.")
else:
    response = requests.get(url)
    if response.status_code == 200:
        local_path.write_bytes(response.content)
        print(f"File downloaded and saved as {local_path}")
    else:
        print(f"Failed to download the file. Status code: {response.status_code}")

Chinook.db already exists, skipping download.


In [16]:
import sqlite3

con = sqlite3.connect("Chinook.db")
cursor = con.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = [row[0] for row in cursor.fetchall() if not row[0].startswith("sqlite_")]

print("Dialect: sqlite")
print(f"Available tables: {tables}")

cursor.execute("SELECT * FROM Artist LIMIT 5;")
print(f"Sample output: {cursor.fetchall()}")
con.close()

Dialect: sqlite
Available tables: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']
Sample output: [(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains')]


In [17]:
import sqlite3
from langchain.tools import tool

# Below are minimal tools for demonstration purposes.
# They are not intended to be secure or for production use.

@tool
def sql_db_list_tables() -> str:
    """Input is an empty string, output is a comma-separated list of tables in the database."""
    con = sqlite3.connect("Chinook.db")
    try:
        cursor = con.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = [row[0] for row in cursor.fetchall() if not row[0].startswith("sqlite_")]
        return ", ".join(tables)
    finally:
        con.close()

@tool
def sql_db_schema(table_names: str) -> str:
    """Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables.
    Be sure that the tables actually exist by calling sql_db_list_tables first!
    Example Input: table1, table2, table3"""
    con = sqlite3.connect("Chinook.db")
    try:
        cursor = con.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        valid_tables = {row[0] for row in cursor.fetchall() if not row[0].startswith("sqlite_")}
        results = []
        for table in table_names.split(","):
            table = table.strip()
            if table not in valid_tables:
                results.append(f"Error: table_names {{{table!r}}} not found in database")
                continue
            cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name=?;", (table,))
            schema_row = cursor.fetchone()
            if schema_row:
                results.append(schema_row[0])
                try:
                    quoted_table = '"' + table.replace('"', '""') + '"'
                    cursor.execute(f"SELECT * FROM {quoted_table} LIMIT 3;")
                    rows = cursor.fetchall()
                    if rows:
                        col_names = [description[0] for description in cursor.description]
                        results.append(f"/*\n3 rows from {table} table:\n" + "\t".join(col_names) + "\n" + "\n".join("\t".join(str(x) for x in row) for row in rows) + "\n*/")
                except Exception as e:
                    results.append(f"Error fetching sample rows: {e}")
        return "\n\n".join(results)
    finally:
        con.close()

@tool
def sql_db_query(query: str) -> str:
    """Input to this tool is a detailed and correct SQL query, output is a result from the database.
    If the query is not correct, an error message will be returned.
    If an error is returned, rewrite the query, check the query, and try again.
    If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields."""
    con = sqlite3.connect("Chinook.db")
    try:
        cursor = con.cursor()
        cursor.execute(query)
        res = cursor.fetchall()
        return str(res)
    except Exception as e:
        return f"Error: {e}"
    finally:
        con.close()

@tool
def sql_db_query_checker(query: str) -> str:
    """Use this tool to double check if your query is correct before executing it.
    Always use this tool before executing a query with sql_db_query!"""
    trigger_prompt = """{query}
Double check the sqlite query above for common mistakes, including:
- Using NOT IN with NULL values
- Using UNION when UNION ALL should have been used
- Using BETWEEN for exclusive ranges
- Data type mismatch in predicates
- Properly quoting identifiers
- Using the correct number of arguments for functions
- Casting to the correct data type
- Using the proper columns for joins

If there are any of the above mistakes, rewrite the query. If there are no mistakes, just reproduce the original query.

Output the final SQL query only.

SQL Query: """.format(query=query)

    response = llm.invoke(trigger_prompt)
    return response.text.strip()

tools = [sql_db_list_tables, sql_db_schema, sql_db_query, sql_db_query_checker]

for tool in tools:
    print(f"{tool.name}: {tool.description}\n")

sql_db_list_tables: Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_schema: Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables.
Be sure that the tables actually exist by calling sql_db_list_tables first!
Example Input: table1, table2, table3

sql_db_query: Input to this tool is a detailed and correct SQL query, output is a result from the database.
If the query is not correct, an error message will be returned.
If an error is returned, rewrite the query, check the query, and try again.
If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

sql_db_query_checker: Use this tool to double check if your query is correct before executing it.
Always use this tool before executing a query with sql_db_query!



In [18]:
system_prompt = """
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

You MUST double check your query before executing it. If you get an error while
executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the
database.

To start you should ALWAYS look at the tables in the database to see what you
can query. Do NOT skip this step.

Then you should query the schema of the most relevant tables.
""".format(
    dialect="sqlite",
    top_k=5,
)

In [19]:
from langchain.agents import create_agent


agent = create_agent(
    llm,
    tools,
    system_prompt=system_prompt,
)

In [ ]:
question = "Which genre on average has the longest tracks?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

give summary of data
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (chatcmpl-tool-8db0c71412bdf311)
 Call ID: chatcmpl-tool-8db0c71412bdf311
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (chatcmpl-tool-95061ba68c59b5c3)
 Call ID: chatcmpl-tool-95061ba68c59b5c3
  Args:
    table_names: Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
================================= Tool Message =================================
Name: sql_db_schema

CREATE TABLE [Album]
(
    [AlbumId] INTEGER  NOT NULL,
    [Title] NVAR

Trace(trace_id=tr-9b0bab5c6b54466e8cca08a5bb3f080d)

In [ ]:
question = "give summary of data"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()